# PILOT — GPT-6.1 Terra — ZERO-SHOT Experiment A

**Run this before the full notebook** (`GPT_ExperimentA_ZeroShot.ipynb`). It makes ~25 real, synchronous calls (condition `zs_A`), checks image handling, prompt format, answer parsing and token usage, and prints a GO / NO-GO verdict with the projected cost of the full 1600-image run.

* Costs a few cents to about a dollar. Nothing is written to your results files.
* Pilot answers are saved to Drive and re-used by the full notebook's smoke test, so nothing is paid for twice.
* Add `OPENAI_API_KEY` in Colab **Secrets** first.

## 1. Mount Drive & confirm input data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
from zipfile import ZipFile

zip_path = "/content/drive/MyDrive/CrowdDensityResearch/Phase5_Input.zip"

if not os.path.exists("/content/Phase5_Input"):
    with ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall("/content/")

print("Input ready:", os.path.exists("/content/Phase5_Input"))

## 2. Results folder (pilot answers are cached here)

In [ ]:
import os

RESULT_DIR = "/content/drive/MyDrive/CrowdDensityResearch/VLM_Results"
os.makedirs(RESULT_DIR, exist_ok=True)

## 3. Install packages

In [ ]:
!pip install -q -U openai

## 4. Connect to GPT-6.1 Terra & configuration
Everything that affects the results is set here. `temperature` / `top_p` are **not** sent (not allowed while reasoning is on), so runs are not strictly deterministic.
Prices are third-party-reported — check them on OpenAI's pricing page. `QUEUE_TOKEN_LIMIT` must match **your tier's** batch queue limit (Build tier = 3M).

In [ ]:
import os, io, re, json, time, math, base64, random, hashlib
from functools import lru_cache
from concurrent.futures import ThreadPoolExecutor, as_completed
from google.colab import userdata
from openai import OpenAI

client = OpenAI(api_key=userdata.get("OPENAI_API_KEY"))

COND = "zs_A"
MODEL            = "gpt-6.1-terra"
REASONING_EFFORT = "medium"      # none | low | medium | high | xhigh | max
IMAGE_DETAIL     = "high"        # low | high | original | auto
MAX_OUTPUT_TOKENS = 25000        # cap incl. reasoning tokens (not a cost)

PRICE_IN, PRICE_OUT = 1.00, 5.00 # USD per 1M tokens — VERIFY
BATCH_DISCOUNT = 0.5
N_PILOT, PILOT_SEED, PILOT_WORKERS = 25, 42, 4

MAX_FILE_MB       = 100          # hard limit is 200 MB per batch file
QUEUE_TOKEN_LIMIT = 3_000_000    # <-- set to your tier's batch queue limit for gpt-6.1-terra
QUEUE_SAFETY      = 0.85

CONFIG = dict(model=MODEL, effort=REASONING_EFFORT, detail=IMAGE_DETAIL, max_out=MAX_OUTPUT_TOKENS, image_max=512, cond=COND)
CONFIG_HASH = hashlib.sha1(json.dumps(CONFIG, sort_keys=True).encode()).hexdigest()[:10]
print(CONFIG, CONFIG_HASH)

## 5. Helpers

In [ ]:
from PIL import Image

def load_image(path, max_size=512):
    """Same resize as the open-source runs."""
    img = Image.open(path).convert("RGB")
    img.thumbnail((max_size, max_size))
    return img

DENSITY_TEXT = {
    0: "Low",
    1: "Medium",
    2: "High"
}

@lru_cache(maxsize=3000)
def img_data_url(path):
    buf = io.BytesIO()
    load_image(path).save(buf, format="JPEG", quality=95)
    return "data:image/jpeg;base64," + base64.b64encode(buf.getvalue()).decode()

def _img(path):
    return {"type": "input_image", "image_url": img_data_url(path), "detail": IMAGE_DETAIL}

## 6. Zero-shot prompt builder — Experiment A
System prompt is the zero-shot version of the open-source notebooks. The builder returns one Responses-API request body (used for both the pilot and the Batch file).

In [ ]:
SYSTEM_PROMPT = """You are an expert in crowd counting and crowd density estimation.

You will receive one image.

Predict:

1. The EXACT crowd count.
2. The crowd density.

Density labels:
0 = Low
1 = Medium
2 = High

Return ONLY:

Count: <integer>
Density: <0/1/2>

Do not explain.
Do not estimate.
Do not output ranges.
Return exactly one integer count.
"""

TEST_TURN = """TEST IMAGE

Count every visible person exactly once.

Return ONLY

Count: <integer>

Density: <0/1/2>
"""


def build_messages_zeroshot_A(test_image):
    """Responses-API request for one test image — zero-shot Experiment A."""
    inp = [{"role": "user", "content": [_img(test_image), {"type": "input_text", "text": TEST_TURN}]}]
    return {"model": MODEL, "instructions": SYSTEM_PROMPT, "input": inp,
            "reasoning": {"effort": REASONING_EFFORT}, "max_output_tokens": MAX_OUTPUT_TOKENS, "store": False}

## 7. Inference and parsing (same robust parser used throughout this project)

In [ ]:
def run_gpt(request, retries=5):
    """One synchronous call (pilot / smoke test only). Returns (response_dict, error_string)."""
    last = None
    for attempt in range(retries):
        try:
            return client.responses.create(**request).model_dump(), None
        except Exception as e:
            last = f"{type(e).__name__}: {e}"
            time.sleep(2 ** attempt)
    return None, last


def parse_response(response):
    """Rejects degenerate long digit-runs as unparseable rather than taking them literally."""
    response = str(response)

    count = None
    m = re.search(r"Count\s*[:=]\s*(\d+)", response, re.IGNORECASE)
    if m:
        digits = m.group(1)
        if len(digits) <= 5:
            count = int(digits)

    density = None
    m = re.search(r"Density\s*[:=]\s*([012])", response, re.IGNORECASE)
    if m:
        density = int(m.group(1))
    else:
        r = response.lower()
        if "low" in r:
            density = 0
        elif "medium" in r:
            density = 1
        elif "high" in r:
            density = 2

    return count, density


def extract_text(body):
    if not body:
        return ""
    parts = []
    for item in body.get("output") or []:
        if item.get("type") == "message":
            for c in item.get("content") or []:
                if c.get("type") == "output_text":
                    parts.append(c.get("text", ""))
    return "".join(parts).strip()


def usage_of(body):
    u = (body or {}).get("usage") or {}
    return dict(in_tok=u.get("input_tokens", 0), out_tok=u.get("output_tokens", 0),
                reason_tok=(u.get("output_tokens_details") or {}).get("reasoning_tokens", 0))


# offline self-test (no API call)
_fake = {"status": "completed", "output": [{"type": "reasoning", "summary": []},
         {"type": "message", "content": [{"type": "output_text", "text": "Count: 123\n\nDensity: 2"}]}],
         "usage": {"input_tokens": 900, "output_tokens": 400, "output_tokens_details": {"reasoning_tokens": 380}}}
assert parse_response(extract_text(_fake)) == (123, 2)
assert parse_response("Count: 12345678\nDensity: 1") == (None, 1)
assert usage_of(_fake) == dict(in_tok=900, out_tok=400, reason_tok=380)
print("Parser self-test passed.")

## 8. Load test data & smoke test (25-image pilot) — READ THIS OUTPUT BEFORE RUNNING THE FULL LOOP
No `support_mapping.csv` needed for zero-shot. Prints the rendered prompt structure, runs 25 random test images with real calls, and reports parse rate, tokens per request, reasoning tokens, MAE and the projected full cost with a GO / NO-GO verdict.

In [ ]:
import pandas as pd
import os

ROOT = "/content/Phase5_Input"
TEST_DIR = f"{ROOT}/test/images"

test_df = pd.read_csv(f"{ROOT}/test.csv")

TEST_FILES = sorted(test_df["filename"].astype(int).tolist())

def test_path(f): return os.path.join(TEST_DIR, f"{int(f):04d}.jpg")

def make_request(f):
    f = int(f)
    return build_messages_zeroshot_A(test_path(f))

# ---- data checks ----
assert len(TEST_FILES) == len(set(TEST_FILES)) == 1600, len(TEST_FILES)
_missing = [test_path(f) for f in TEST_FILES if not os.path.exists(test_path(f))]
assert not _missing, f"{len(_missing)} image files missing, e.g. {_missing[:3]}"
print("Data checks passed:", len(TEST_FILES), "test images")

In [ ]:
def render(req):
    out = ["[instructions] " + req["instructions"].split("\n")[0] + " ..."]
    for m in req["input"]:
        if isinstance(m["content"], str):
            out.append(f"[{m['role']}] {m['content']!r}")
        else:
            parts = ["<image>" if c["type"] == "input_image" else c["text"] for c in m["content"]]
            out.append(f"[{m['role']}] " + " | ".join(repr(p) for p in parts))
    return "\n".join(out)

_req = make_request(TEST_FILES[0])
_n_img = sum(1 for m in _req["input"] if not isinstance(m["content"], str) for c in m["content"] if c["type"] == "input_image")
assert _n_img == 1, _n_img
assert "temperature" not in _req and _req["reasoning"] == {"effort": REASONING_EFFORT}
assert ("YOLO predicted count: " in render(_req)) == False
print(render(_req))
print("\nImages in request:", _n_img, "| request size:", round(len(json.dumps(_req)) / 1024), "KB")

In [ ]:
PILOT_PATH = f"{RESULT_DIR}/gpt_pilot_{COND}_{CONFIG_HASH}.jsonl"
PILOT_FILES = sorted(random.Random(PILOT_SEED).sample(TEST_FILES, N_PILOT))

pilot = {}
if os.path.exists(PILOT_PATH):
    for line in open(PILOT_PATH, encoding="utf-8"):
        d = json.loads(line); pilot[d["fname"]] = d
print("Resumed from disk:", len(pilot))

def _one(f):
    body, err = run_gpt(make_request(f))
    return {"fname": int(f), "body": body, "err": err}

todo = [f for f in PILOT_FILES if f not in pilot]
with open(PILOT_PATH, "a", encoding="utf-8") as out, ThreadPoolExecutor(PILOT_WORKERS) as ex:
    for d in ex.map(_one, todo):
        pilot[d["fname"]] = d
        out.write(json.dumps(d) + "\n"); out.flush()
print("Pilot calls stored:", len(pilot))

def row_from_body(fname, body, err=None):
    text = extract_text(body)
    pc, pd_ = parse_response(text)
    row = dict(test_filename=int(fname), predicted_count=pc, predicted_density=pd_, raw_response=text,
               status=(body or {}).get("status"),
               incomplete=((body or {}).get("incomplete_details") or {}).get("reason"),
               error=err, **usage_of(body))
    pass
    return row

P = pd.DataFrame([row_from_body(d["fname"], d["body"], d["err"]) for d in pilot.values()])
gt = test_df.set_index("filename")
P["true_count"] = P["test_filename"].map(gt["total_count"])
P["true_density"] = P["test_filename"].map(gt["crowd_density"])
P["cost_batch"] = (P.in_tok * PRICE_IN + P.out_tok * PRICE_OUT) / 1e6 * BATCH_DISCOUNT

ok = P.dropna(subset=["predicted_count"])
PILOT_MEAN_IN = float(P.in_tok.mean())
print(f"\nPilot ({len(P)} images)  [{COND}]")
print("  count parsed   :", f"{100 * P.predicted_count.notna().mean():.0f}%")
print("  density parsed :", f"{100 * P.predicted_density.notna().mean():.0f}%")
print("  API errors     :", int(P.error.notna().sum()), "| incomplete:", int(P.incomplete.notna().sum()))
print("  tokens/request : in", round(P.in_tok.mean()), "| out", round(P.out_tok.mean()), "| reasoning", round(P.reason_tok.mean()), "| max out", int(P.out_tok.max()))
print("  pilot MAE      :", round((ok.predicted_count - ok.true_count).abs().mean(), 1), "| density acc:", f"{100 * (P.predicted_density == P.true_density).mean():.0f}%")

print('  input tokens (1 image + prompt):', round(P.in_tok.mean()), ' <- a 512px image should be roughly 230-310 tokens (inference, not confirmed for Terra)')
proj = P.cost_batch.mean() * len(TEST_FILES)
print(f"  projected full run: ~${proj:.2f} on Batch (~${proj / BATCH_DISCOUNT:.2f} standard)")

problems = []
if P.error.notna().any(): problems.append("API errors")
if P.incomplete.notna().any(): problems.append("incomplete responses -> raise MAX_OUTPUT_TOKENS")
if P.predicted_count.isna().mean() > 0.02: problems.append("count parse rate < 98%")
if P.predicted_density.isna().mean() > 0.02: problems.append("density parse rate < 98%")
print("\nVERDICT:", "NO-GO -> " + "; ".join(problems) if problems else "GO (no blocking problems in the pilot)")
print(P[["test_filename", "true_count", "predicted_count", "true_density", "predicted_density", "reason_tok", "raw_response"]].head(5).to_string())

In [ ]:
P.to_csv(f"{RESULT_DIR}/gpt_pilot_report_{COND}.csv", index=False)
print("Saved pilot report:", f"{RESULT_DIR}/gpt_pilot_report_{COND}.csv")

## Next step
If the verdict above is **GO**, open `GPT_ExperimentA_ZeroShot.ipynb`, and run it top to bottom. Set `CONFIRM_FULL_RUN = True` in section 9a only when you are ready to spend the projected amount.
If **NO-GO**, fix what the verdict lists (for example raise `MAX_OUTPUT_TOKENS` for incomplete responses) and re-run the pilot.